# GNN6종_모델설명

각 설명 다음의 코드 셀을 Shift+Enter로 실행하세요. 이 노트북은 함수를 정의하고 내부 모듈을 준비합니다. 실제 전처리·학습 호출은 **00번 통합 노트북**에서 합니다. 내부 파일은 셀에서 자동 생성하므로 별도 Python 파일을 준비할 필요가 없습니다.

**모델은 총 6종입니다:** TGNN, GAT, GATv2, GINE, Edge GraphSAGE, Edge-weighted GCN. 아래에 각각 `모델 1/6`부터 `모델 6/6`까지 설명과 코드 셀이 있습니다. `TransactionGNN`은 6종이 공유하는 거래 분류 구조이며 별도의 일곱 번째 모델이 아닙니다.

In [ ]:
from pathlib import Path
import sys, json
# 통합 노트북은 이 폴더를 작업 폴더로 지정합니다. 개별 실행도 같은 폴더에서 하세요.
WORKSPACE = Path.cwd().resolve()
RUNTIME = WORKSPACE / '_runtime'
(RUNTIME / 'gnn_pipeline').mkdir(parents=True, exist_ok=True)
(RUNTIME / 'gnn_pipeline' / '__init__.py').touch()
if str(RUNTIME) not in sys.path:
    sys.path.insert(0, str(RUNTIME))


## 실행 셀을 내부 모듈로 연결

아래 함수는 이 노트북의 코드 셀을 그대로 저장해 다음 단계에서 import할 수 있게 합니다.

In [ ]:
def export_cells(notebook_name, module_name):
    """이 노트북의 실제 코드 셀을 모아 내부 import용 모듈을 생성합니다."""
    notebook = json.loads((WORKSPACE / notebook_name).read_text(encoding='utf-8'))
    blocks = [cell['source'] for cell in notebook['cells']
              if cell['cell_type'] == 'code' and cell.get('metadata', {}).get('module') == module_name]
    source = '\n\n'.join(''.join(block) if isinstance(block, list) else block for block in blocks) + '\n'
    path = RUNTIME / 'gnn_pipeline' / (module_name + '.py')
    compile(source, str(path), 'exec')
    path.write_text(source, encoding='utf-8')
    print('내부 모듈 준비:', path.name)
    return path


## 6개 모델의 차이

| 모델 | 이웃 정보를 모으는 방식 |
|---|---|
| TGNN | Transformer 어텐션. 거래 특징을 중요도와 전달 정보에 반영 |
| GAT | 이웃별 어텐션. 거래 특징을 중요도에 반영 |
| GATv2 | GAT와 어텐션 계산 순서를 바꾼 구조 |
| GINE | 거래 특징을 더한 이웃 메시지 합산 후 신경망 적용 |
| Edge GraphSAGE | 변환한 이웃+거래 메시지의 평균 |
| Edge-weighted GCN | 거래 특징으로 연결 가중치를 학습한 GCN |

6종 모두 송신·수신 계좌 표현과 대상 거래 특징으로 거래 한 건을 분류합니다. GNN 2층 + 분류기를 공유합니다. GraphSAGE와 GCN은 엣지 특징을 반영한 확장형입니다. 클러스터링은 별도의 MiniBatch K-means가 수행하며 특정 GNN을 대신 사용하는 단계가 아닙니다.

## 라이브러리·상수 준비

다음 함수가 사용하는 라이브러리와 설정을 준비합니다.

In [ ]:
"""Six edge-aware classifiers sharing the same two-layer transaction head."""
import torch
from torch import nn
from torch_geometric.nn import GATConv, GATv2Conv, GCNConv, GINEConv, MessagePassing, TransformerConv

MODEL_NAMES = ('tgnn', 'gat', 'gatv2', 'gine', 'edge_sage', 'edge_weighted_gcn')



## `EdgeSAGE`

이웃 계좌와 거래 특징을 변환해 더한 메시지의 평균을 구하고 자기 계좌 정보를 더합니다.

**메서드:** `__init__`, `forward`, `message`. 클래스 내부 메서드는 같은 상태를 공유하므로 함께 정의합니다.

In [ ]:
class EdgeSAGE(MessagePassing):
    """Average transformed neighbors plus transaction attributes; retain the root."""
    def __init__(self, inputs, edges, hidden):
        super().__init__(aggr='mean')
        self.neighbor = nn.Linear(inputs, hidden, bias=False)
        self.edge = nn.Linear(edges, hidden, bias=False)
        self.root = nn.Linear(inputs, hidden)

    def forward(self, x, edge_index, edge_attr):
        messages = self.propagate(edge_index, x=self.neighbor(x),
                                  edge_attr=self.edge(edge_attr), size=(len(x), len(x)))
        return self.root(x) + messages

    def message(self, x_j, edge_attr):
        return x_j + edge_attr

## `EdgeWeightedGCN`

거래 특징을 0~2 범위의 연결 가중치로 바꿔 GCN에 전달하고 자기 계좌 정보를 더합니다.

**메서드:** `__init__`, `forward`. 클래스 내부 메서드는 같은 상태를 공유하므로 함께 정의합니다.

In [ ]:
class EdgeWeightedGCN(nn.Module):
    """Convert transaction attributes to positive GCN edge weights."""
    def __init__(self, inputs, edges, hidden):
        super().__init__()
        self.conv = GCNConv(inputs, hidden, add_self_loops=False, cached=False)
        self.edge_weight = nn.Linear(edges, 1)
        self.root = nn.Linear(inputs, hidden, bias=False)

    def forward(self, x, edge_index, edge_attr):
        weights = 2 * torch.sigmoid(self.edge_weight(edge_attr).squeeze(-1))
        return self.conv(x, edge_index, edge_weight=weights) + self.root(x)

## 모델 1/6 — TGNN

Transformer 어텐션으로 이웃과 거래 정보를 반영합니다. 이 프로젝트의 TGNN은 시간 메모리 TGN이 아닌 Graph Transformer입니다.

**입력:** `inputs`는 계좌 입력 차원, `edges`는 거래 피처 수, `hidden`은 출력 차원, `heads`는 어텐션 헤드 수입니다. 어텐션을 사용하지 않는 모델도 공통 호출을 위해 같은 인자를 받습니다.

**반환:** `TransformerConv` 계층 하나. 공통 `TransactionGNN`이 이 생성 함수를 두 번 호출해 GNN 두 층을 구성합니다.

**실행:** 아래 셀은 이 모델의 생성 함수를 정의하며, 마지막 확인 셀에서 실제 모델 생성과 예측을 실행합니다.

In [ ]:
def make_tgnn_layer(inputs, edges, hidden, heads):
    """Create one TGNN layer."""
    if hidden % heads:
        raise ValueError("hidden must be divisible by heads")
    return TransformerConv(inputs, hidden // heads, heads=heads, edge_dim=edges, dropout=0.)


## 모델 2/6 — GAT

이웃별 중요도를 어텐션으로 학습합니다. 거래 특징을 중요도 계산에 반영하고 자기 계좌 정보는 residual 경로로 전달합니다.

**입력:** `inputs`는 계좌 입력 차원, `edges`는 거래 피처 수, `hidden`은 출력 차원, `heads`는 어텐션 헤드 수입니다. 어텐션을 사용하지 않는 모델도 공통 호출을 위해 같은 인자를 받습니다.

**반환:** `GATConv` 계층 하나. 공통 `TransactionGNN`이 이 생성 함수를 두 번 호출해 GNN 두 층을 구성합니다.

**실행:** 아래 셀은 이 모델의 생성 함수를 정의하며, 마지막 확인 셀에서 실제 모델 생성과 예측을 실행합니다.

In [ ]:
def make_gat_layer(inputs, edges, hidden, heads):
    """Create one GAT layer."""
    if hidden % heads:
        raise ValueError("hidden must be divisible by heads")
    return GATConv(inputs, hidden // heads, heads=heads, edge_dim=edges,
                   dropout=0., add_self_loops=False, residual=True)


## 모델 3/6 — GATv2

GAT와 다른 어텐션 계산 순서를 사용합니다. GAT와는 별개의 계층이며 별개의 모델로 학습합니다.

**입력:** `inputs`는 계좌 입력 차원, `edges`는 거래 피처 수, `hidden`은 출력 차원, `heads`는 어텐션 헤드 수입니다. 어텐션을 사용하지 않는 모델도 공통 호출을 위해 같은 인자를 받습니다.

**반환:** `GATv2Conv` 계층 하나. 공통 `TransactionGNN`이 이 생성 함수를 두 번 호출해 GNN 두 층을 구성합니다.

**실행:** 아래 셀은 이 모델의 생성 함수를 정의하며, 마지막 확인 셀에서 실제 모델 생성과 예측을 실행합니다.

In [ ]:
def make_gatv2_layer(inputs, edges, hidden, heads):
    """Create one GATv2 layer."""
    if hidden % heads:
        raise ValueError("hidden must be divisible by heads")
    return GATv2Conv(inputs, hidden // heads, heads=heads, edge_dim=edges,
                     dropout=0., add_self_loops=False, residual=True)


## 모델 4/6 — GINE

거래 특징을 포함한 이웃 메시지를 합산한 뒤 신경망으로 변환합니다. 자기 정보 비중을 조절하는 epsilon도 학습합니다.

**입력:** `inputs`는 계좌 입력 차원, `edges`는 거래 피처 수, `hidden`은 출력 차원, `heads`는 어텐션 헤드 수입니다. 어텐션을 사용하지 않는 모델도 공통 호출을 위해 같은 인자를 받습니다.

**반환:** `GINEConv` 계층 하나. 공통 `TransactionGNN`이 이 생성 함수를 두 번 호출해 GNN 두 층을 구성합니다.

**실행:** 아래 셀은 이 모델의 생성 함수를 정의하며, 마지막 확인 셀에서 실제 모델 생성과 예측을 실행합니다.

In [ ]:
def make_gine_layer(inputs, edges, hidden, heads):
    """Create one GINE layer."""
    if hidden % heads:
        raise ValueError("hidden must be divisible by heads")
    network = nn.Sequential(nn.Linear(inputs, hidden), nn.ReLU(), nn.Linear(hidden, hidden))
    return GINEConv(network, edge_dim=edges, train_eps=True)


## 모델 5/6 — Edge GraphSAGE

위에서 정의한 EdgeSAGE를 사용합니다. 이웃 표현과 거래 특징을 더한 메시지를 평균 내고 자기 정보를 더합니다.

**입력:** `inputs`는 계좌 입력 차원, `edges`는 거래 피처 수, `hidden`은 출력 차원, `heads`는 어텐션 헤드 수입니다. 어텐션을 사용하지 않는 모델도 공통 호출을 위해 같은 인자를 받습니다.

**반환:** `EdgeSAGE` 계층 하나. 공통 `TransactionGNN`이 이 생성 함수를 두 번 호출해 GNN 두 층을 구성합니다.

**실행:** 아래 셀은 이 모델의 생성 함수를 정의하며, 마지막 확인 셀에서 실제 모델 생성과 예측을 실행합니다.

In [ ]:
def make_edge_sage_layer(inputs, edges, hidden, heads):
    """Create one Edge GraphSAGE layer."""
    if hidden % heads:
        raise ValueError("hidden must be divisible by heads")
    return EdgeSAGE(inputs, edges, hidden)


## 모델 6/6 — Edge-weighted GCN

위에서 정의한 EdgeWeightedGCN을 사용합니다. 거래 특징으로 양수 연결 가중치를 만든 뒤 GCN 집계에 반영합니다.

**입력:** `inputs`는 계좌 입력 차원, `edges`는 거래 피처 수, `hidden`은 출력 차원, `heads`는 어텐션 헤드 수입니다. 어텐션을 사용하지 않는 모델도 공통 호출을 위해 같은 인자를 받습니다.

**반환:** `EdgeWeightedGCN` 계층 하나. 공통 `TransactionGNN`이 이 생성 함수를 두 번 호출해 GNN 두 층을 구성합니다.

**실행:** 아래 셀은 이 모델의 생성 함수를 정의하며, 마지막 확인 셀에서 실제 모델 생성과 예측을 실행합니다.

In [ ]:
def make_edge_weighted_gcn_layer(inputs, edges, hidden, heads):
    """Create one Edge-weighted GCN layer."""
    if hidden % heads:
        raise ValueError("hidden must be divisible by heads")
    return EdgeWeightedGCN(inputs, edges, hidden)


## 6종 선택 함수 — `make_layer`

입력한 모델 이름에 해당하는 위의 생성 함수를 호출합니다. 새로운 모델이 아니라 6종을 공통 학습 코드에 연결하는 함수입니다.

In [ ]:
def make_layer(name, inputs, edges, hidden, heads):
    """Select one of six independently defined model-layer factories."""
    factories = {
        'tgnn': make_tgnn_layer,
        'gat': make_gat_layer,
        'gatv2': make_gatv2_layer,
        'gine': make_gine_layer,
        'edge_sage': make_edge_sage_layer,
        'edge_weighted_gcn': make_edge_weighted_gcn_layer,
    }
    if name not in factories:
        raise ValueError(f'Unknown model: {name}')
    return factories[name](inputs, edges, hidden, heads)


## `TransactionGNN`

GNN 두 층으로 계좌 표현을 만든 뒤 송신·수신 계좌와 거래 특징을 합쳐 거래별 logit을 계산합니다.

**메서드:** `__init__`, `forward`. 클래스 내부 메서드는 같은 상태를 공유하므로 함께 정의합니다.

In [ ]:
class TransactionGNN(nn.Module):
    """Predict one logit per target transaction, not per account."""
    def __init__(self, name, nodes=19, edges=17, hidden=128, heads=2):
        super().__init__()
        self.conv1 = make_layer(name, nodes, edges, hidden, heads)
        self.conv2 = make_layer(name, hidden, edges, hidden, heads)
        self.head = nn.Sequential(nn.Linear(2 * hidden + edges, hidden), nn.ReLU(), nn.Linear(hidden, 1))

    def forward(self, x, edge_index, edge_attr, targets, target_attr):
        hidden = torch.relu(self.conv1(x, edge_index, edge_attr))
        hidden = torch.relu(self.conv2(hidden, edge_index, edge_attr))
        transaction = torch.cat([hidden[targets[0]], hidden[targets[1]], target_attr], dim=1)
        return self.head(transaction).squeeze(-1)

## 다음 단계에 연결

실행한 코드 셀을 내부 모듈로 저장합니다. 내용을 수정했다면 이 셀을 다시 실행하고 통합 노트북의 커널을 재시작하세요.

In [ ]:
export_cells('30_GNN6종_모델설명.ipynb', 'models')

## 실제로 6종 생성·예측 확인

작은 임의 그래프에서 모델별로 거래 2건을 예측합니다. 아래 표가 **6행**이면 6종 모두 생성되고 실행된 것입니다. 실제 거래 학습이나 성능 평가는 수행하지 않습니다. 내부 모듈을 새로 불러오므로 위 셀을 수정했다면 내보내기 셀도 다시 실행하세요.

In [ ]:
import importlib
import pandas as pd
from gnn_pipeline import models as model_module
model_module = importlib.reload(model_module)
torch.set_num_threads(2)
x = torch.zeros(4, 19)
edge_index = torch.tensor([[0, 1, 2, 3], [1, 2, 3, 0]], dtype=torch.long)
edge_attr = torch.zeros(4, 17)
targets = torch.tensor([[0, 1], [1, 2]], dtype=torch.long)
target_attr = torch.zeros(2, 17)
checks = []
for name in model_module.MODEL_NAMES:
    model = model_module.TransactionGNN(name, hidden=16, heads=2).eval()
    with torch.no_grad():
        logits = model(x, edge_index, edge_attr, targets, target_attr)
    assert logits.shape == (2,) and torch.isfinite(logits).all()
    checks.append({'모델': name, '실제 계층': type(model.conv1).__name__, '거래 출력 개수': len(logits), '실행 성공': True})
assert len(checks) == 6
display(pd.DataFrame(checks))